In [82]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import requests
import pathlib
import json

In [33]:
from dotenv import load_dotenv
import os
load_dotenv() 
# print(type(os.getenv("VISUAL_CROSSING_API_KEY")))

<class 'str'>


In [55]:
def getDataFromOpenMeteo(latitude, longitude, startDate, endDate, fileName):
  # Data Source 1
	# Setup the Open-Meteo API client with cache and retry on error
	cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
	retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
	openmeteo = openmeteo_requests.Client(session = retry_session)

	# Make sure all required weather variables are listed here
	# The order of variables in hourly or daily is important to assign them correctly below
	url = "https://archive-api.open-meteo.com/v1/archive"
	params = {
		"latitude": latitude,
		"longitude": longitude,
		"start_date": startDate,
		"end_date": endDate,
		"daily": ["temperature_2m_max", "temperature_2m_min", "sunshine_duration", "precipitation_hours", "wind_speed_10m_max"],
	}
	responses = openmeteo.weather_api(url, params=params)

	# Process first location. Add a for-loop for multiple locations or weather models
	response = responses[0]
	print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation {response.Elevation()} m asl")
	print(f"Timezone {response.Timezone()} {response.TimezoneAbbreviation()}")
	print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

	# Process daily data. The order of variables needs to be the same as requested.
	daily = response.Daily()
	daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
	daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
	daily_sunshine_duration = daily.Variables(2).ValuesAsNumpy()
	daily_precipitation_hours = daily.Variables(3).ValuesAsNumpy()
	daily_wind_speed_10m_max = daily.Variables(4).ValuesAsNumpy()

	daily_data = {"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	)}
	daily_data["temperature_2m_max"] = daily_temperature_2m_max
	daily_data["temperature_2m_min"] = daily_temperature_2m_min
	daily_data["sunshine_duration"] = daily_sunshine_duration
	daily_data["precipitation_hours"] = daily_precipitation_hours
	daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max

	daily_dataframe = pd.DataFrame(data = daily_data)
	daily_dataframe.to_csv("openMeteo_" + '_'.join([fileName, startDate, 'to', endDate]) +  ".csv", index=False)
	return daily_dataframe
	# print(daily_dataframe)


In [75]:
latitude = [40.79736, 41.78701, 30.1444, 25.7738]
longitude = [-73.97785, -87.77166, -97.66876, -80.1936]
cities = ["ny", "il", "tx", "fl"]
start_date = "2016-01-01"
end_date = "2024-03-12"
daily_data = []
for i in range(len(latitude)):
  daily_data.append(getDataFromOpenMeteo(latitude[i], longitude[i], start_date, end_date, cities[i]))

print(len(daily_data))

Coordinates 40.808433532714844°N -74.0198974609375°E
Elevation 0.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Coordinates 41.79261779785156°N -87.78262329101562°E
Elevation 187.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Coordinates 30.123022079467773°N -97.67523193359375°E
Elevation 154.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Coordinates 25.764497756958008°N -80.19607543945312°E
Elevation 7.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
4


In [20]:
ny_data = daily_data[0]

In [28]:
# convert pandas._libs.tslibs.timestamps.Timestamp to datetime.date
ny_data['date'] = ny_data['date'].apply(lambda x: x.date())

In [29]:
ny_data['date'][0]

datetime.date(2016, 1, 1)

In [84]:
def getDataFromVisualCrossing(latitude, longitude, startDate, endDate, fileName):
  url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/" + str (latitude) + \
        "%2C" + str(longitude) + "/" + startDate + "/" + endDate + "?unitGroup=us&include=days&key="+ os.getenv("VISUAL_CROSSING_API_KEY") + "&contentType=json"
  print(url)
  print("https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/40.79736%2C-73.97785/2016-01-01/today?unitGroup=us&include=days&key=MHFU2QHX7NTY5RTWZPAT7VBXS&contentType=json")
# "https://weather.visualcrossing.com/VisualCrosingWebServices/rest/services/timeline/
# 40.79736%2C-73.97785/2016-01-01/today?unitGroup=us&include=days&key=MHFU2QHX7NTY5RTWZPAT7VBXS&contentType=json"


  payload={}
  headers = {}

  response = requests.request("GET", url, headers=headers, data=payload)
  pathlib.Path("visualCrossing_" + '_'.join([fileName, startDate, 'to', endDate]) + '.json').write_bytes(response.content)

  print(response.text)

In [80]:
# WARNING!!!
# Will incur an API cost, don't re-run
# Historical Data is saved to a CSV
# visual_crossing_data = []
# for i in range(len(latitude)):
  # daily_data.append(
  # visual_crossing_data.append(getDataFromVisualCrossing(latitude[i], longitude[i], start_date, end_date, cities[i]))

In [83]:
def readStoredVisualCrossingData(fileName):
  with open(fileName, 'r') as file:
    # Reading from json file
    data = json.load(file)
  return data

In [87]:
# Read all visual crossing files
vc_data = []
for i in range(len(latitude)):
  fileName = "visualCrossing_" + '_'.join([cities[i], start_date, 'to', end_date]) + '.json'
  vc_data.append(readStoredVisualCrossingData(fileName))

In [93]:
type(vc_data)

list

In [99]:
for cityData in vc_data:
  city_df = pd.DataFrame(cityData['days'])
  city_df = city_df[['datetime', 'tempmax', 'tempmin', 'humidity', 'windspeed']]
  print(city_df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2994 entries, 0 to 2993
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   datetime   2994 non-null   object 
 1   tempmax    2994 non-null   float64
 2   tempmin    2994 non-null   float64
 3   humidity   2994 non-null   float64
 4   windspeed  2994 non-null   float64
dtypes: float64(4), object(1)
memory usage: 117.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2994 entries, 0 to 2993
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   datetime   2994 non-null   object 
 1   tempmax    2994 non-null   float64
 2   tempmin    2994 non-null   float64
 3   humidity   2994 non-null   float64
 4   windspeed  2994 non-null   float64
dtypes: float64(4), object(1)
memory usage: 117.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2994 entries, 0 to 2993
Data columns (total 5 columns):
 #  